In [ ]:
#!pip install openai-agents -q

In [ ]:
!uvx --version
!uv --version

### Setup

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown

from agents import Agent, Runner
from agents.mcp import MCPServerStdio

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise ValueError("OPENAI_API_KEY environment variable not set")

MODEL = "gpt-4.1-mini"

In [ ]:
WEB_AGENT_PROMPT = """
You are a helpful assistant.
When the user asks about a web page, if you can access the internet, then read the page first, then answer based on what you actually read.
Always cite the URL you read from in the format [source: <URL>].
If you cannot access the internet, then say so.
"""

In [ ]:
web_agent = Agent(
    name="Web Agent",
    instructions=WEB_AGENT_PROMPT,
    model=MODEL
)

In [ ]:
result = await Runner.run(
    web_agent,
    input="Who is the guest mentioned on https://www.superdatascience.com/999",
    max_turns=10
)

print(f"Last Agent: {result.last_agent.name}")
print("-----")
display(Markdown(result.final_output))

In [ ]:
result = await Runner.run(
    web_agent,
    input="Read https://news.ycombinator.com/ and give me the titles of the top 5 stories",
    max_turns=10
)

print(f"Last Agent: {result.last_agent.name}")
print("-----")
display(Markdown(result.final_output))

### Launch our first MCP Server

#### Step 1: Describe Launch Parameters

In [ ]:
fetch_server_params = {
    "command": "uvx",
    "args": ["mcp-server-fetch"]
}

#### Let's Investigate The Server

In [ ]:
async with MCPServerStdio(name="Fetch Server", params=fetch_server_params, client_session_timeout_seconds=60) as server:
    tools = await server.list_tools()

    print(f"✅ Connected. The server offers {len(tools)} tools(s):\n")
    for tool in tools:
        print(f"🔧 - {tool.name}: {tool.description}")

#### Step 2: Launch & Connect to the server

In [ ]:
async with MCPServerStdio(name="Fetch Server", params=fetch_server_params, client_session_timeout_seconds=60) as server:
    web_agent = Agent(
        name="Web Agent",
        instructions=WEB_AGENT_PROMPT,
        model=MODEL,
        mcp_servers = [server]
    )

    result = await Runner.run(
        web_agent,
        input="Who is the guest mentioned on https://www.superdatascience.com/podcast/sds-999-whats-left-to-build-when-software-is-free-with-chip-huyen",
        max_turns=30
    )

print(f"Last Agent: {result.last_agent.name}")
print("-----")
display(Markdown(result.final_output))

In [ ]:
async with MCPServerStdio(name="Fetch Server", params=fetch_server_params, client_session_timeout_seconds=60) as server:
    web_agent = Agent(
        name="Web Agent",
        instructions=WEB_AGENT_PROMPT,
        model=MODEL,
        mcp_servers = [server]
    )

    result = await Runner.run(
        web_agent,
        input="Read https://news.ycombinator.com/ and give me the titles of the top 5 stories",
        max_turns=30
    )

print(f"Last Agent: {result.last_agent.name}")
print("-----")
display(Markdown(result.final_output))